스마트 해운물류 x AI 미션 챌린지 : 이상신호 감지 기반 비정상 작동 진단

https://dacon.io/competitions/official/236590/overview/description

# 수상작 리뷰


---

### 1. 코드 흐름 요약 (Pipeline)
1. **데이터 로드 및 탐색 (EDA):** 도메인 의미가 가려진 비식별화 데이터(`X_01`, `X_02` 등)의 분포 확인 및 정상/비정상(Target) 클래스의 불균형(Imbalance) 비율 파악.
2. **피처 엔지니어링 (Feature Engineering):** 센서 데이터들의 기초 통계량(Mean, Std, Max, Min 등) 파생 변수 생성 및 상관계수(Correlation)가 높은 변수 간의 사칙연산 조합 도출.
3. **데이터 전처리 (Preprocessing):** 센서값의 스케일 차이를 맞추기 위한 스케일링(Standard/Robust) 및 학습을 방해하는 극단적인 이상치(Outlier) 처리.
4. **모델링 (Modeling):** 정형 데이터 분류에 강력한 트리 기반 앙상블 모델(LightGBM, XGBoost, CatBoost 등) 구축.
5. **교차 검증 및 튜닝 (CV & Tuning):** K-Fold 교차 검증을 통해 과적합 방지 및 Optuna 등을 이용한 하이퍼파라미터 최적화.
6. **앙상블 및 추론 (Ensemble & Inference):** 여러 Fold와 모델에서 도출된 예측 확률의 평균(Soft Voting)을 구하여 최종 비정상 여부 예측.

### 2. 새롭게 알게 된 내용
* **블랙박스(비식별화) 데이터 다루기:** 도메인 지식이 없는 센서 데이터라도 Feature Importance나 SHAP Value를 활용하면 모델이 중요하게 생각하는 핵심 피처를 역추적하고, 의미 있는 파생 변수를 수학적으로 만들어낼 수 있음을 알게 되었습니다.
* **클래스 불균형 해결 기법:** 비정상 데이터가 적은 이상 탐지 상황에서 무작정 SMOTE 같은 오버샘플링을 쓰기보다, 트리 모델 내부의 가중치 파라미터(`scale_pos_weight`, `is_unbalance`)를 조정하거나 Stratified K-Fold를 사용하는 것이 종종 더 안정적이고 효과적이라는 점을 배웠습니다.

### 3. 어려웠던 내용
* **과적합(Overfitting) 제어:** 피처를 무작위로 조합하고 늘리다 보니 로컬 검증 점수는 오르는데 실제 리더보드 점수는 하락하는 과적합 현상을 제어하는 것이 까다로웠습니다. 모델 성능에 기여하는 유효한 피처만 남기는 피처 셀렉션(Feature Selection) 과정에 많은 고민이 필요했습니다.
* **미세한 이상 패턴 포착:** 센서 값의 변화가 극단적이지 않은 '미세한 이상'을 잡아내기 위해, 센서 간의 상호작용(Interaction)을 어떻게 수치화할지 결정하는 과정이 어려웠습니다.

### 4. 배울 점
* **탄탄한 CV(Cross Validation) 전략의 중요성:** 단순히 Public 리더보드 점수에 흔들리지 않고, 본인만의 안정적인 로컬 CV(교차 검증) 점수를 구축하고 믿어야 최종 순위 변동(Shake-up)을 방어할 수 있음을 체감했습니다.
* **단일 모델보다는 앙상블(Ensemble):** 단일 모델 하나를 영혼까지 튜닝하는 것보다, 서로 다른 특성을 가진 모델(LGBM, XGBoost, RF 등)의 다양성(Diversity)을 확보하여 앙상블하는 것이 일반화 성능 향상에 훨씬 강력함을 배웠습니다.

### 5. 주요 코드 정리 (Compact)

```python
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import StratifiedKFold

# 1. 피처 엔지니어링 (비식별화 센서 데이터 통계치 파생 변수 생성)
sensor_features = [col for col in train.columns if col.startswith('X_')]
train['sensor_mean'] = train[sensor_features].mean(axis=1)
train['sensor_std'] = train[sensor_features].std(axis=1)
test['sensor_mean'] = test[sensor_features].mean(axis=1)
test['sensor_std'] = test[sensor_features].std(axis=1)

# 학습에 사용할 변수 설정 (ID 등 불필요 컬럼 제외)
features = train.columns.drop(['ID', 'Target'])

# 2. 교차 검증 및 모델 학습 설정
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = []

for train_idx, val_idx in skf.split(train[features], train['Target']):
    X_tr, y_tr = train[features].iloc[train_idx], train['Target'].iloc[train_idx]
    X_val, y_val = train[features].iloc[val_idx], train['Target'].iloc[val_idx]
    
    # 클래스 불균형을 고려한 트리 모델 생성 (is_unbalance=True)
    model = LGBMClassifier(n_estimators=1000, is_unbalance=True, random_state=42)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(stopping_rounds=100, verbose=False)]
    )
    models.append(model)

# 3. 앙상블 및 최종 추론 (Soft Voting)
preds = []
for model in models:
    # 비정상(1) 클래스에 대한 예측 확률 추출
    pred_prob = model.predict_proba(test[features])[:, 1]
    preds.append(pred_prob)

# 5개 Fold 예측 확률의 평균 계산
final_prob = np.mean(preds, axis=0)

# 확률 임계값 0.5를 기준으로 정상(0)/비정상(1) 분류
test['Target'] = np.where(final_prob > 0.5, 1, 0)
```